In [ ]:
# Phase (I): Test GRPO on Game of 24 — surface drawbacks D1-D4
#
#   D1  CoT redundancy           → track CoT length distribution across training
#   D2  No per-token signal      → (structural; visualised by absence of v_t)
#   D3  Zero-pass@K dead zone    → curate hard puzzles, track which never solve
#   D4  Indiscriminate credit    → inspect failed rollouts that share good prefixes
#
# Phase (II) [later]: swap reward + advantage for velocity / answer-buffer / prefix-buffer
#                     and re-run the same diagnostics to verify the four fixes.

In [1]:
import os, re, json, random, itertools, math
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
random.seed(0); np.random.seed(0); torch.manual_seed(0)

MODEL_NAME    = "Qwen/Qwen3-0.6B"
OUTPUT_DIR    = Path("output/game24_grpo_baseline")
ROLLOUT_LOG   = OUTPUT_DIR / "rollouts.jsonl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR.resolve())

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_ref" in ModelCreateRes has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_output" in EvalResultsTrial has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/pyd

Output dir: /Users/fangyuanyu/Implementation/arl/output/game24_grpo_baseline


/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_configuration" in DynamicLeaderboardConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


## 2. Game-of-24 &nbsp;·&nbsp; verifier &nbsp;·&nbsp; solver &nbsp;·&nbsp; difficulty buckets

The verifier parses an expression, checks each of the 4 input numbers is used
exactly once with `+ - * /`, and that it evaluates to 24.

We also enumerate every solvable 4-tuple so we can bucket puzzles by difficulty
(number of distinct solutions). Hard puzzles (1-2 solutions) feed the **D3**
zero-pass@K test.

In [2]:
TARGET = 24
EPS    = 1e-6
ALLOWED = set("0123456789+-*/(). ")

def safe_eval(expr: str) -> Optional[float]:
    """Evaluate a numeric expression restricted to digits, + - * / ( ) and spaces."""
    if not expr or any(c not in ALLOWED for c in expr):
        return None
    try:
        return eval(expr, {"__builtins__": {}}, {})
    except Exception:
        return None

def verify_24(numbers: List[int], expr: str) -> bool:
    """True iff `expr` uses each integer in `numbers` exactly once and evaluates to 24."""
    val = safe_eval(expr)
    if val is None or abs(val - TARGET) > EPS:
        return False
    used = [int(x) for x in re.findall(r"\d+", expr)]
    return sorted(used) == sorted(numbers)

# Enumerate solutions for a given 4-tuple (used for difficulty bucketing and
# for *populating the answer buffer* in Phase II).
def enumerate_solutions(numbers: Tuple[int, ...], max_solutions: int = 20) -> List[str]:
    sols = set()
    ops = ["+", "-", "*", "/"]
    for perm in set(itertools.permutations(numbers)):
        a, b, c, d = perm
        for op1, op2, op3 in itertools.product(ops, repeat=3):
            for tmpl in [
                "(({a}{o1}{b}){o2}{c}){o3}{d}",
                "({a}{o1}({b}{o2}{c})){o3}{d}",
                "({a}{o1}{b}){o2}({c}{o3}{d})",
                "{a}{o1}(({b}{o2}{c}){o3}{d})",
                "{a}{o1}({b}{o2}({c}{o3}{d}))",
            ]:
                expr = tmpl.format(a=a, b=b, c=c, d=d, o1=op1, o2=op2, o3=op3)
                if verify_24(list(numbers), expr):
                    sols.add(expr)
                    if len(sols) >= max_solutions:
                        return list(sols)
    return list(sols)

# Sanity check
print("solutions for (3,4,5,6):", enumerate_solutions((3, 4, 5, 6))[:3])
print("verify ((3+5)*(7-4)):", verify_24([3, 5, 7, 4], "(3+5)*(7-4)"))
print("verify wrong:        ", verify_24([3, 5, 7, 4], "3+5+7+4"))

solutions for (3,4,5,6): ['6*((3+5)-4)', '((5+3)-4)*6', '((5-4)+3)*6']
verify ((3+5)*(7-4)): True
verify wrong:         False


In [3]:
# Build the dataset: all solvable 4-tuples from 1..9, bucketed by # solutions.
def build_puzzle_pool(max_n: int = 9) -> List[Dict[str, Any]]:
    pool = []
    for tup in itertools.combinations_with_replacement(range(1, max_n + 1), 4):
        sols = enumerate_solutions(tup, max_solutions=50)
        if sols:
            pool.append({"numbers": list(tup), "solutions": sols, "n_solutions": len(sols)})
    return pool

puzzles = build_puzzle_pool(9)
print(f"Solvable 4-tuples (digits 1-9, with repetition): {len(puzzles)}")

# Bucket by difficulty
easy   = [p for p in puzzles if p["n_solutions"] >= 8]    # many solutions
medium = [p for p in puzzles if 3 <= p["n_solutions"] < 8]
hard   = [p for p in puzzles if p["n_solutions"] <= 2]    # for D3 zero-pass@K test

print(f"  easy   (≥8 sols): {len(easy)}")
print(f"  medium (3-7 sols): {len(medium)}")
print(f"  hard   (≤2 sols): {len(hard)}")
print("example hard puzzle:", hard[0] if hard else None)

Solvable 4-tuples (digits 1-9, with repetition): 404
  easy   (≥8 sols): 336
  medium (3-7 sols): 44
  hard   (≤2 sols): 24
example hard puzzle: {'numbers': [1, 2, 7, 7], 'solutions': ['((7*7)-1)/2'], 'n_solutions': 1}


In [4]:
# Train split: 200 mixed puzzles. Hold out 30 hard puzzles for the D3 probe.
random.shuffle(easy); random.shuffle(medium); random.shuffle(hard)

train_puzzles = easy[:80] + medium[:80] + hard[:40]
random.shuffle(train_puzzles)

eval_puzzles      = easy[80:100] + medium[80:100]
hard_probe        = hard[40:70]    # for the D3 dead-zone test (held out, NEVER trained)

print(f"train={len(train_puzzles)}, eval={len(eval_puzzles)}, hard_probe={len(hard_probe)}")

SYSTEM_PROMPT = (
    "You play the Game of 24. Given four numbers, write a single arithmetic "
    "expression using each number exactly once with + - * / and parentheses "
    "that evaluates to 24. Put the final expression on the last line after "
    "'#### '. Example: '#### (3+5)*(7-4)'."
)

def to_chat(p):
    user = f"Numbers: {p['numbers']}. Make 24."
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user + " /no_think"},
        ],
        "numbers":   p["numbers"],
        "solutions": p["solutions"],
    }

train_ds = Dataset.from_list([to_chat(p) for p in train_puzzles])
eval_ds  = Dataset.from_list([to_chat(p) for p in eval_puzzles])
probe_ds = Dataset.from_list([to_chat(p) for p in hard_probe])
print(train_ds[0])

train=148, eval=20, hard_probe=0
{'prompt': [{'content': "You play the Game of 24. Given four numbers, write a single arithmetic expression using each number exactly once with + - * / and parentheses that evaluates to 24. Put the final expression on the last line after '#### '. Example: '#### (3+5)*(7-4)'.", 'role': 'system'}, {'content': 'Numbers: [3, 3, 8, 8]. Make 24. /no_think', 'role': 'user'}], 'numbers': [3, 3, 8, 8], 'solutions': ['8/(3-(8/3))']}


## 3. Rewards &nbsp;·&nbsp; correctness, format

Standard GRPO setup: trajectory-level rewards only (D2 is structural).

In [5]:
def _text(completion) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return completion[0].get("content", "")
    return str(completion)

def extract_expr(text: str) -> str:
    m = re.search(r"####\s*(.+?)\s*$", text.strip())
    return m.group(1).strip() if m else ""

def correctness_reward(completions, numbers, **kwargs):
    rewards = []
    for c, nums in zip(completions, numbers):
        rewards.append(1.0 if verify_24(list(nums), extract_expr(_text(c))) else 0.0)
    return rewards

def format_reward(completions, **kwargs):
    return [0.2 if re.search(r"####\s*\S", _text(c)) else 0.0 for c in completions]

# Sanity
fake = [[{"role": "assistant", "content": "Let me think...\n#### (3+5)*(7-4)"}]]
print("correctness:", correctness_reward(fake, numbers=[[3, 5, 7, 4]]))
print("format    :", format_reward(fake))

correctness: [1.0]
format    : [0.2]


## 4. Rollout recorder &nbsp;·&nbsp; the diagnostic backbone

This is the *single* hook that powers all four D1–D4 diagnostics. It piggybacks
on the reward-function callback so we get every rollout the trainer sees,
without modifying TRL. Each rollout gets logged with its puzzle, completion,
extracted expression, correctness flag, and token length.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class RolloutLogger:
    """Logs every rollout to JSONL via the reward-fn callback. Returns 0.0 reward."""
    def __init__(self, path: Path, tok):
        self.path = path
        self.tok  = tok
        self.step = 0
        self.path.write_text("")  # truncate

    def __call__(self, completions, numbers, solutions=None, **kwargs):
        with self.path.open("a") as f:
            for i, (c, nums) in enumerate(zip(completions, numbers)):
                text = _text(c)
                expr = extract_expr(text)
                correct = verify_24(list(nums), expr)
                n_tok = len(self.tok.encode(text, add_special_tokens=False))
                f.write(json.dumps({
                    "step": self.step,
                    "idx": i,
                    "numbers": list(nums),
                    "completion": text,
                    "expr": expr,
                    "correct": bool(correct),
                    "n_tokens": int(n_tok),
                }) + "\n")
        self.step += 1
        return [0.0] * len(completions)   # passthrough, contributes nothing

    # Keep TRL happy: it inspects __name__ on reward callables.
    __name__ = "rollout_logger"

rollout_logger = RolloutLogger(ROLLOUT_LOG, tokenizer)
print("Rollout log →", ROLLOUT_LOG)

Rollout log → output/game24_grpo_baseline/rollouts.jsonl


## 5. Train &nbsp;·&nbsp; vanilla GRPO

Short run (200 steps, 8 generations/prompt) is enough to surface the four
drawbacks. Bump `max_steps` for a longer collapse-watch.

In [ ]:
config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_generations=8,
    max_completion_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=200,
    logging_steps=5,
    bf16=True,
    save_strategy="no",
    report_to="none",
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.4,
)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward, rollout_logger],
    args=config,
    train_dataset=train_ds,
)
trainer.train()
print("Done. Rollouts at:", ROLLOUT_LOG)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


: 

## 6. Diagnostics — surfacing D1, D3, D4

We load the rollout log and compute four lenses. D2 is structural (there is no
per-token signal to plot) — its absence is the diagnostic.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rollouts = [json.loads(l) for l in ROLLOUT_LOG.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rollouts)
df["key"] = df["numbers"].apply(lambda x: tuple(sorted(x)))
print(f"{len(df)} rollouts logged across {df['step'].nunique()} reward-fn calls")
df.head(3)

In [ ]:
# ── D1: CoT length over training (overall + split by correct/incorrect) ──
agg = df.groupby("step").agg(
    n_tokens_mean=("n_tokens", "mean"),
    n_tokens_p90=("n_tokens", lambda s: np.percentile(s, 90)),
    acc=("correct", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(agg.step, agg.n_tokens_mean, label="mean")
axes[0].plot(agg.step, agg.n_tokens_p90,  label="p90", linestyle="--")
axes[0].set_xlabel("reward-fn call (~training step)"); axes[0].set_ylabel("CoT tokens")
axes[0].set_title("D1 · CoT length over training"); axes[0].legend()

for ok, label in [(True, "correct"), (False, "incorrect")]:
    sub = df[df.correct == ok].groupby("step").n_tokens.mean()
    axes[1].plot(sub.index, sub.values, label=label)
axes[1].set_xlabel("step"); axes[1].set_ylabel("mean tokens")
axes[1].set_title("D1 · length, split by correctness"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── D3: zero-pass@K dead-zone — which puzzles never get solved? ──
solved_per_puzzle = df.groupby("key").correct.any()
n_solved   = int(solved_per_puzzle.sum())
n_attempted = int(len(solved_per_puzzle))
print(f"Puzzles ever solved during training: {n_solved}/{n_attempted}  "
      f"({n_solved/n_attempted:.1%})")

# Cross-reference with the # of solutions per puzzle
puzzle_info = pd.DataFrame([
    {"key": tuple(sorted(p["numbers"])), "n_solutions": p["n_solutions"]}
    for p in train_puzzles
]).drop_duplicates("key").set_index("key")
joined = puzzle_info.join(solved_per_puzzle.rename("ever_solved"))
bucketed = joined.groupby(pd.cut(joined.n_solutions, [0, 2, 7, 100],
                                  labels=["hard (≤2)", "med (3-7)", "easy (≥8)"])).ever_solved.agg(["mean", "count"])
print("\nPass-rate by difficulty bucket:")
print(bucketed)
print("\nD3 signature: hard-bucket pass-rate stays at ~0 → these puzzles "
      "receive no learning signal because GRPO has no positive trajectory "
      "in their group.")

In [ ]:
# ── D4: indiscriminate credit — failed rollouts that share good prefixes ──
# For each puzzle that had *both* successes and failures in the same group,
# find the longest common prefix between failed and successful completions.
def lcp(a: str, b: str) -> int:
    n = 0
    for x, y in zip(a, b):
        if x != y: break
        n += 1
    return n

shared = []
for (step, key), g in df.groupby(["step", "key"]):
    succ = g[g.correct]; fail = g[~g.correct]
    if len(succ) == 0 or len(fail) == 0:
        continue
    for _, f in fail.iterrows():
        best = max(lcp(f.completion, s.completion) for _, s in succ.iterrows())
        shared.append({"step": step, "fail_len": len(f.completion),
                       "shared_prefix": best,
                       "prefix_frac": best / max(1, len(f.completion))})
sdf = pd.DataFrame(shared)
print(f"Failed rollouts sharing a prefix with a correct sibling: {len(sdf)}")
if len(sdf):
    print(f"  median shared-prefix fraction: {sdf.prefix_frac.median():.2%}")
    print(f"  mean   shared-prefix fraction: {sdf.prefix_frac.mean():.2%}")
    plt.figure(figsize=(6, 3))
    plt.hist(sdf.prefix_frac, bins=30)
    plt.xlabel("shared-prefix fraction (failed vs correct sibling)")
    plt.ylabel("# rollouts")
    plt.title("D4 · failed rollouts that share productive prefixes\n"
              "GRPO discriminates against ALL these tokens equally")
    plt.tight_layout(); plt.show()

In [ ]:
# ── D2: structural — there is no per-token signal to plot. ──
# GRPO broadcasts a single trajectory advantage to every token in the rollout.
# Demonstrate this by printing a single rollout with its (constant) per-token advantage.
sample = df.sample(1, random_state=0).iloc[0]
adv = float(sample.correct)  # standin: 1.0 for correct, 0.0 for incorrect
toks = tokenizer.tokenize(sample.completion)
print(f"Puzzle: {sample.numbers}   correct={sample.correct}   "
      f"|tokens|={len(toks)}")
print(f"per-token advantage (GRPO): a constant {adv:.2f} for ALL {len(toks)} tokens")
print(f"→ this is the structural D2 problem the velocity reward will fix.")
print("\nFirst 30 tokens (all assigned the same advantage):")
print("  " + " ".join(toks[:30]))

## 7. Summary &nbsp;·&nbsp; what we expect to see

| Drawback | Expected signature in the plots above |
|---|---|
| **D1** | CoT mean/p90 length grows or stays high; incorrect rollouts especially verbose |
| **D2** | (structural) — single broadcast advantage per trajectory |
| **D3** | Hard-bucket pass-rate stays near 0% throughout training |
| **D4** | Many failed rollouts share large prefixes (>30%) with successful siblings — yet every token gets equal negative credit |

Phase (II) will introduce the velocity reward + answer/prefix buffers and
re-run this same diagnostic suite to verify each row flips.